# Swept Pazy wing static deflection sweep

Run a grid of swept Pazy cases with varying AoA and dynamic pressure in parallel using JAX's `pmap`.

Set XLA flags to set one host device per case. This must run before JAX is imported anywhere in the kernel, otherwise JAX will initialise the default device count and ignore these flags.

In [ ]:
import os

n_vel = 32  # number of velocities
n_alpha = 6  #number of angles of attack
n_case = n_vel * n_alpha
os.environ["XLA_FLAGS"] = (
        os.environ.get("XLA_FLAGS", "")
        + f" --xla_force_host_platform_device_count={n_case} --xla_cpu_multi_thread_eigen=true"
).strip()

Imports

In [ ]:
import time
from typing import cast

import jax
from jax import Array
from jax import numpy as jnp
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize

from flapjax.aero.flowfields import ConstantFlowField
from flapjax.coupled import CoupledAeroelastic
from flapjax.models.pazy.swept.swept_pazy_wing import generate_swept_pazy_wing
from flapjax.utils.print_utils import set_verbosity

JAX configuration and verbosity

In [ ]:
set_verbosity("silent")  # suppress console prints as there would be a lot for 1k+ cases

# ensure that the XLA flags have worked
assert jax.device_count() == n_case, (
    f"Expected {n_case} devices, got {jax.device_count()}"
)

Build the AoA / velocity sweep and stack the individual `CoupledAeroelastic` cases into a single pytree with a leading device axis.

In [ ]:
rho = 1.225  # freestream density
u_inf_vec = jnp.linspace(1e-7, 80.0, n_vel)  # freestream velocity sweep
alphas = jnp.deg2rad(jnp.array((0.0, 1.0, 3.0, 5.0, 7.0, 10.0)))  # angle of attack sweep

# create a list of all the cases. This means creating an instance of the swept Pazy wing for each condition.
cases = []
for u_inf in u_inf_vec:
    u_inf_mag = u_inf
    for alpha in alphas:
        cases.append(
            generate_swept_pazy_wing(
                flowfield=ConstantFlowField(
                    u_inf=jnp.array(
                        (
                            u_inf_mag,
                            0.0,
                            0.0,
                        )
                    ),
                    rho=rho,
                    relative_motion=True,
                ),
                gravity=jnp.array((0.0, -9.81, 0.0)),
                aoa=alpha,
                m=12,
                node_multiplier=2,
                sweep_angle=20,
                tip_mass=None
            )
        )

# stack the list of cases into a JAX pytree so that we can map across them
stacked_case = jax.tree_util.tree_map(lambda *xs: jnp.stack(xs), *cases)

Per-case solve function. Returns the mid-chord tip Z displacement normalised by the semi-span.

In [ ]:
def solve(case_: CoupledAeroelastic) -> tuple[Array, Array]:
    static_sol = case_.static_solve(
        prescribed_dofs=tuple(range(6)),  # prescribe the 6 degress of freedom at the root (clamped)
        horseshoe=True,  # use a horseshoe wake as this means only one wake panel per strip
    )

    # compute the mid-chord deflection from the aerodynamic grid coordinate output zeta_b
    tip_z = 0.5 / 0.55 * (
            static_sol.aero.zeta_b[0][0, -1, 2]
            + static_sol.aero.zeta_b[0][-1, -1, 2]
    )

    tip_norm = static_sol.aero.nc[0][0, -1, :]
    tip_norm /= jnp.linalg.norm(tip_norm)

    tip_alpha = jnp.asin(-tip_norm[0])

    return tip_z, tip_alpha


parallel_func = jax.pmap(solve)  # create a function that maps the solve function across the stacked cases

Run the analysis for all cases in parallel and time the execution. The results are reshaped into a grid for plotting.

In [ ]:
t_start = time.time()
tip_z_, tip_alpha_ = parallel_func(stacked_case)  # evaluate the function
jax.block_until_ready([tip_z_, tip_alpha_])
t_end = time.time()

print("Total time: ", t_end - t_start)
print("Time per case: ", (t_end - t_start) / n_case)

tip_z_grid = tip_z_.reshape(n_vel, n_alpha).T  # [n_alpha, n_vel] for plotting
tip_alpha_grid = tip_alpha_.reshape(n_vel, n_alpha).T

# subtract initial discrepancy in z (for an undeformed swept wing with aoa, there will be a nonzero tip z)
tip_z_grid -= tip_z_grid[:, [0]]

Data from from "Flutter, Post-Flutter, and Limit-Cycle Oscillations of Very Flexible Swept Wings", Fig. 6a) and 8a)

In [ ]:
tip_z_alpha_0 = jnp.array(((229.81700753498387, 0.001310615989515096),
                           (538.751345532831, 0.004587155963302725),
                           (987.0828848223898, 0.00917431192660545),
                           (1484.3918191603875, 0.008519003931848013),
                           (2219.0527448869752, 0.017693315858453462),
                           (3032.831001076426, 0.01245085190039319)))
tip_z_alpha_1 = jnp.array(((229.81700753498387, 0.005242463958060273),
                           (572.6587728740583, 0.01834862385321101),
                           (1009.6878363832077, 0.03145478374836175),
                           (1522.066738428418, 0.04325032765399739),
                           (2139.935414424112, 0.047837483617300114)))
tip_z_alpha_3 = jnp.array(((233.83620689655174, 0.019659239842726106),
                           (569.5043103448276, 0.04914809960681521),
                           (980.603448275862, 0.07011795543905636),
                           (1527.4784482758619, 0.08781127129750982),
                           (2172.413793103448, 0.10615989515072083),
                           (2975.7543103448274, 0.1205766710353866)))
tip_z_alpha_5 = jnp.array(((264.00862068965523, 0.041284403669724745),
                           (531.7887931034483, 0.07142857142857145),
                           (991.9181034482762, 0.11009174311926606),
                           (1561.4224137931033, 0.15006553079947577),
                           (2191.271551724138, 0.17758846657929228),
                           (2889.008620689655, 0.19921363040629098)))
tip_z_alpha_7 = jnp.array(((260.2370689655172, 0.05701179554390562),
                           (558.1896551724138, 0.09960681520314546),
                           (984.3750000000001, 0.15399737876802094),
                           (1508.6206896551726, 0.1959370904325033),
                           (2127.155172413793, 0.23525557011795545),
                           (2968.2112068965516, 0.281127129750983)))
tip_z_alpha_10 = jnp.array(((245.15086206896558, 0.07404980340760159),
                            (561.961206896552, 0.14613368283093053),
                            (980.603448275862, 0.21231979030144166),
                            (1455.8189655172414, 0.26474442988204455),
                            (2100.7543103448274, 0.3217562254259502)))
tip_z_plot_data = (tip_z_alpha_0, tip_z_alpha_1, tip_z_alpha_3, tip_z_alpha_5, tip_z_alpha_7, tip_z_alpha_10)

In [ ]:
tip_aoa_alpha_0 = jnp.array(((539.4629780309194, 0.1025641025641022),
                             (986.167615947925, -0.02564102564102555),
                             (1481.6924328722537, -0.06410256410256387),
                             (2218.877135882831, -0.2948717948717956),
                             (3034.174125305126, -0.21794871794871717)))
tip_aoa_alpha_1 = jnp.array(((231.89585028478433, 1.1282051282051277),
                             (571.1960943856794, 1.0),
                             (1010.5777054515866, 0.6282051282051277),
                             (1523.1895850284784, 0.2820512820512828),
                             (2140.7648494711143, 0.14102564102564052)))
tip_aoa_alpha_3 = jnp.array(((236.77786818551664, 2.846153846153846),
                             (568.7550854353132, 2.294871794871794),
                             (981.2855980471927, 1.9358974358974361),
                             (1525.6305939788447, 1.5641025641025639),
                             (2172.4979658258744, 1.2051282051282044),
                             (2973.1489015459724, 1.0)))
tip_aoa_alpha_5 = jnp.array(((263.6289666395443, 4.448717948717948),
                             (532.1399511798209, 3.8717948717948714),
                             (991.0496338486573, 3.115384615384615),
                             (1557.3637103336046, 2.7051282051282044),
                             (2189.585028478438, 1.7948717948717938),
                             (2885.272579332791, 1.4487179487179471)))
tip_aoa_alpha_7 = jnp.array(((258.74694873881197, 6.282051282051282),
                             (556.5500406834825, 5.487179487179487),
                             (986.167615947925, 4.6410256410256405),
                             (1508.5435313262813, 3.6794871794871797),
                             (2126.1187957689176, 2.7692307692307683),
                             (2965.8258746948736, 2.1923076923076916)))
tip_aoa_alpha_10 = jnp.array(((244.10089503661516, 9.051282051282051),
                              (561.4320585842148, 7.448717948717949),
                              (981.2855980471927, 6.230769230769231),
                              (1454.841334418226, 5.217948717948718),
                              (2099.26769731489, 4.102564102564102)))

tip_aoa_plot_data = (tip_aoa_alpha_0, tip_aoa_alpha_1, tip_aoa_alpha_3, tip_aoa_alpha_5, tip_aoa_alpha_7,
                     tip_aoa_alpha_10)

Plot of wingtip deflection versus root AoA, colour-coded by angle of attack.

In [ ]:
alphas_deg = jnp.rad2deg(alphas)

# noinspection PyArgumentList
norm = cast(Normalize, Normalize(vmin=float(alphas_deg.min()), vmax=float(alphas_deg.max())))

cmap = plt.get_cmap("viridis")

q_inf_vec = 0.5 * 1.225 * u_inf_vec ** 2

fig, ax = plt.subplots()
for i_aoa, aoa in enumerate(alphas_deg):
    ax.plot(q_inf_vec, tip_z_grid[i_aoa, :], color=cmap(norm(float(aoa))))

for data, aoa in zip(tip_z_plot_data, [0.0, 1.0, 3.0, 5.0, 7.0, 10.0]):
    ax.scatter(data[:, 0], data[:, 1], marker="x", color=cmap(norm(aoa)))

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
fig.colorbar(sm, ax=ax, label="Root angle of Attack [deg]")
ax.plot([], [], color="b", label="Computational")
ax.scatter([], [], color="b", marker="x", label="Experimental")
ax.legend()
ax.set_xlabel("Dynamic pressure [Pa]")
ax.set_ylabel("Tip Z Displacement [z/b]")
ax.set_title("Swept Pazy (20 deg) Wingtip Z Displacement")
plt.show()


Plot of wingtip angle of attack versus root AoA, colour-coded by angle of attack.

In [ ]:
fig, ax = plt.subplots()
for i_aoa, aoa in enumerate(alphas_deg):
    ax.plot(q_inf_vec, jnp.rad2deg(tip_alpha_grid[i_aoa]), color=cmap(norm(float(aoa))))

for data, aoa in zip(tip_aoa_plot_data, [0.0, 1.0, 3.0, 5.0, 7.0, 10.0]):
    ax.scatter(data[:, 0], data[:, 1], marker="x", color=cmap(norm(aoa)))

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
fig.colorbar(sm, ax=ax, label="Root angle of Attack [deg]")
ax.plot([], [], color="b", label="Computational")
ax.scatter([], [], color="b", marker="x", label="Experimental")
ax.legend()
ax.set_xlabel("Dynamic pressure [Pa]")
ax.set_ylabel("Tip angle of attack [deg]")
ax.set_title("Swept Pazy (20 deg) Wingtip Angle of Attack")
plt.show()